# 02 · Exploratory Data Analysis (EDA): MIMIC (cleaned)

Reproduces the main statistics and figures of **Task 5** from the Task 4 cleaned metadata
(`data/processed/mimic_cleaned.csv`).

* **Read-only.** The cleaned CSV and the raw images are never modified. Figures are drawn into a temporary folder, not into the repository.
* All logic lives in `src/data/eda.py` (statistics) and `src/data/eda_plots.py` (figures). This notebook only calls them.
* **EDA-only tokenization** (`eda.eda_tokens`) lowercases and splits into letter runs *for counting*. It is **not** the project's preprocessing.
* Script and language indicators are **heuristic**.
* The full pipeline (all tables + figures) is `python scripts/run_eda.py`; the write-up is `docs/EDA_REPORT.md`.

No dataset text is printed in this notebook; only aggregates and sample IDs are shown.

In [ ]:
import sys, json, tempfile
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "config.yaml").exists():   # works whether launched from repo root or notebooks/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, display
from src.data import eda, eda_plots as plots
from src.utils.config import load_config

pd.set_option("display.max_columns", 20)
config = load_config()
names = eda.class_names(config)
FIG = Path(tempfile.mkdtemp(prefix="eda_nb_"))   # figures go here, not into the repository
show = lambda p: display(Image(filename=str(p), width=640))
names

## 1. Verify the input is exactly the Task 4 output

In [ ]:
df = eda.load_cleaned(config)
fingerprint = json.loads((ROOT / "results/metrics/mimic_cleaning_fingerprint.json").read_text(encoding="utf-8"))
checks = eda.verify_cleaned(df, config, fingerprint)
assert all(checks.values()), checks
print(f"{len(df):,} samples; all input checks passed:")
checks

## 2. Class distribution

In [ ]:
class_table, ratio = eda.class_distribution(df, names)
display(class_table)
print(ratio)
show(plots.class_distribution(class_table, FIG / "class_distribution.png"))

The classes are almost balanced (ratio about 1.01), so accuracy is not badly distorted by imbalance. Macro-F1 and per-class metrics remain the primary metrics.

## 3. Secondary labels (misogyny categories)

In [ ]:
secondary = eda.secondary_label_stats(df, config["dataset"]["secondary_label_columns"])
display(secondary["per_label"])
display(secondary["cooccurrence_within_positive"])
show(plots.secondary_combinations(secondary["combinations"], FIG / "secondary.png", int((df.target_label == 1).sum())))

## 4. Text length (original text, non-empty only)

In [ ]:
features = eda.text_features(df)          # computed in memory; df is unchanged
display(eda.length_statistics(features, names))
present = features[features.has_text]
show(plots.histogram(present.char_count, FIG / "chars.png", "Text length (characters)", "Characters per text", bins=50))
show(plots.length_by_class(features, names, FIG / "length_by_class.png"))

## 5. Vocabulary (EDA-only tokenization)

In [ ]:
display(pd.DataFrame(eda.vocabulary_summary(df, names)).T)
uni = eda.top_terms(df, names, n=1, top=30)
display(uni[uni.group == "all"].head(20))
show(plots.top_terms(uni[uni.group == "all"], FIG / "top_terms.png", "Most frequent terms (all samples)"))

Raw frequency is dominated by function words (*hai, me, to, ki…*). The weighted log-odds below compares the classes while accounting for frequency. The z-scores are descriptive, not significance tests.

In [ ]:
distinctive = eda.distinctive_terms(df, names)
display(distinctive.head(10)); display(distinctive.tail(10))
show(plots.distinctive_terms(distinctive, names, FIG / "distinctive.png"))

## 6. Script / Hinglish indicators (HEURISTIC)

In [ ]:
scripts = eda.script_by_class(features, names)
display(scripts)
print("Devanagari share of letters (non-empty texts):", eda.describe(present.devanagari_letter_share.dropna()))
show(plots.grouped_percent_by_class(scripts, "script_category", names, FIG / "scripts.png",
                                    "Script / language indicators by class (HEURISTIC)", "Share of class (%)"))

## 7. Social-media text features

In [ ]:
social = eda.feature_by_class(features, names)
display(social)
show(plots.grouped_percent_by_class(social[(social.filter(like="count_").sum(axis=1)) > 0], "feature", names,
                                    FIG / "social.png", "Text features by class", "Share of samples with text (%)"))

## 8. Images (header metadata only; images are not altered)

In [ ]:
meta = eda.image_metadata(df)
display(eda.image_statistics(meta, names))
display(eda.common_resolutions(meta, names))
for col, title in [("width", "Image width"), ("height", "Image height"), ("aspect_ratio", "Image aspect ratio")]:
    show(plots.histogram(meta[col], FIG / f"{col}.png", title, col, bins=50))

## 9. Text/image availability for fair model comparison

In [ ]:
eda.availability(df)

## 10. Leakage signals

In [ ]:
handles = eda.mention_handles(df, names)
display(handles)
show(plots.handles_by_class(handles, names, FIG / "handles.png"))
print("File-name pattern perfectly predicts label:", eda.id_pattern_association(df)["pattern_perfectly_predicts_label"])

In [ ]:
# Near-duplicate images: dHash on an in-memory grayscale thumbnail (about 20 s; nothing is written)
near = eda.near_duplicate_images(df, max_distance=4)
print(len(near), "candidate pairs;", int(near.cross_class.sum()), "cross-class;",
      int((~near.already_grouped_in_task4).sum()), "not grouped by Task 4")
near.head(10)

## 11. Outliers (recorded, not removed)

In [ ]:
out = eda.outliers(features, meta)
out.groupby("category").agg(samples=("sample_id", "size"), rule=("rule", "first"))

## Summary

See `docs/EDA_REPORT.md`. Each finding there is separated into a measured result, an interpretation and a possible implication. Key points for later tasks:

* **Leakage risks:**
  * label-encoding file names;
  * class-exclusive `@handles` (watermarks);
  * class-correlated image resolutions;
  * near-duplicate images not yet grouped.
* **Text:** 148 samples have no text (106 misogynistic), and about 30% of texts contain Devanagari.
